# Featuresmith Tutorial: 07 — Custom Rules and Advanced Extensions

Learn how to extend `BaseRule` to create custom quality rules, register them in `RuleRegistry`, and execute them via `RuleEngine` or `fs.analyze()`.

---


## 1. Custom Rule Engineering
Featuresmith is designed to be fully extensible. While built-in rules cover standard statistical quality and leakage checks, business-specific constraints (e.g., negative balances, invalid transaction ranges, proprietary schema rules) can be implemented by inheriting from `BaseRule`.

### The `BaseRule` Interface
Custom rules implement six required properties/methods:
- `id`: Unique dot-separated rule identifier (e.g. `custom.zero_variance`).
- `name`: Human-readable title.
- `description`: Summary of rule check.
- `category`: Category string (`quality`, `statistical`, `leakage`, or `custom`).
- `severity`: Default severity (`critical`, `warning`, `info`).
- `enabled_by_default`: Boolean flag.
- `evaluate(profile: ProfileResult) -> list[RuleFinding]`: Evaluation logic consuming a precomputed `ProfileResult`.

### Prerequisite: Prepare the Sales Dataset
This notebook loads `examples/data/processed/sales.csv`, which `examples/prepare_datasets.py` generates deterministically (no network). From the repository root, run:

```bash
python examples/prepare_datasets.py
```


### Step 1: Implement a Custom `ZeroVarianceRule`

In [1]:
import os

import featuresmith as fs
from featuresmith.core.profile_result import ProfileResult
from featuresmith.core.rule_finding import RuleFinding
from featuresmith.rules.base import BaseRule


class ZeroVarianceRule(BaseRule):
    """Detect numeric columns with zero standard deviation."""

    @property
    def id(self) -> str:
        return "custom.zero_variance"

    @property
    def name(self) -> str:
        return "Zero Variance Numeric Columns"

    @property
    def description(self) -> str:
        return "Flags numeric columns with zero observed standard deviation."

    @property
    def category(self) -> str:
        return "custom"

    @property
    def severity(self) -> str:
        return "warning"

    @property
    def enabled_by_default(self) -> bool:
        return True

    def evaluate(self, profile: ProfileResult) -> list[RuleFinding]:
        findings: list[RuleFinding] = []
        for col_name, num_prof in profile.numeric_profiles.items():
            if num_prof.std_dev == 0.0:
                findings.append(
                    RuleFinding(
                        rule_id=self.id,
                        rule_name=self.name,
                        category=self.category,
                        severity=self.severity,
                        column_name=col_name,
                        title="Zero Variance Detected",
                        description=f"Numeric column '{col_name}' has standard deviation of 0.0.",
                        evidence={"std_dev": num_prof.std_dev},
                    )
                )
        return findings


print("Custom ZeroVarianceRule defined successfully.")

Custom ZeroVarianceRule defined successfully.


### Step 2: Register & Evaluate Custom Rule Against Profile

In [2]:
from featuresmith.rules.registry import default_registry

# Instantiate the custom rule and add it to a RuleRegistry
custom_rule = ZeroVarianceRule()
registry = default_registry()
registry.register(custom_rule)

data_path = os.path.join("..", "data", "processed", "sales.csv")
dataset = fs.load(data_path)
profile = fs.profile(dataset)

# Evaluate the custom rule directly
findings = custom_rule.evaluate(profile)

print(f"Rule ID             : {custom_rule.id}")
print(f"Registered Rules    : {len(registry.list_rules())}")
print(f"Direct Findings     : {len(findings)}")
for f in findings:
    print(f"  - [{f.severity.upper()}] Column: {f.column_name} | {f.title}")

Rule ID             : custom.zero_variance
Registered Rules    : 9
Direct Findings     : 0


### Key Takeaways
- `BaseRule` allows developers to add custom, business-specific quality checks.
- Custom rules operate deterministically on precomputed `ProfileResult` descriptors.
- `RuleRegistry` maintains registered rule instances for execution engines.